# Preparação do ambiente no Google Colab

Execute as células em ordem antes de abrir os notebooks de avaliação. Este roteiro usa o TensorFlow já fornecido pelo runtime sempre que possível e não instala automaticamente `requirements.txt`.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

import tensorflow as tf

gpus = tf.config.list_physical_devices("GPU")
print("Python:", sys.version)
print("TensorFlow:", tf.__version__)
print("GPUs:", gpus)

if not gpus:
    raise RuntimeError(
        "Nenhuma GPU foi detectada. Conecte o notebook a um runtime Colab "
        "com GPU e execute novamente."
    )

for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

print("GPU configurada com crescimento dinâmico de memória.")

In [ ]:
repo_dir = Path("/content/replicating_RePO")
repo_url = "https://github.com/marquesgbr/replicating_RePO.git"

if not repo_dir.exists():
    subprocess.run(["git", "clone", repo_url, str(repo_dir)], check=True)
else:
    print(f"Repositório já existe: {repo_dir}")

code_dir = repo_dir / "code"
os.chdir(code_dir)
if str(code_dir) not in sys.path:
    sys.path.insert(0, str(code_dir))

print("Diretório de trabalho:", Path.cwd())
print("Módulo de compatibilidade:", (code_dir / "load_model_compat.py").exists())

In [ ]:
%pip install -q pandas numpy ipykernel

In [ ]:
import importlib.metadata as metadata
import load_model_compat

print("TensorFlow:", metadata.version("tensorflow"))
print("Keras:", metadata.version("keras"))
print("load_model_compat: importado com sucesso")

required_paths = [
    repo_dir / "models" / "pkt_model" / "saved_model.pb",
    repo_dir / "models" / "flw_model" / "saved_model.pb",
]
missing = [str(path) for path in required_paths if not path.exists()]
if missing:
    raise FileNotFoundError("Arquivos de modelo ausentes:\n" + "\n".join(missing))
print("SavedModels encontrados.")

In [ ]:
packet_model = load_model_compat.load_model("../models/pkt_model/")
flow_model = load_model_compat.load_model("../models/flw_model/")

packet_input = tf.zeros((1, 20, 29), dtype=tf.float32)
flow_input = tf.zeros((1, 77), dtype=tf.float32)
packet_output = packet_model(packet_input, training=False)
flow_output = flow_model(flow_input, training=False)

print("Saída do modelo de pacotes:", packet_output.shape, packet_output.device)
print("Saída do modelo de fluxos:", flow_output.shape, flow_output.device)

if "/GPU:" not in packet_output.device or "/GPU:" not in flow_output.device:
    print("Aviso: confirme o runtime e a ocupação com !nvidia-smi.")
else:
    print("Modelos executados na GPU.")

## Próximos passos

Depois de executar todas as células, abra os notebooks na ordem: `Flow_based_RePO_Normal_Test.ipynb`, `Packet_based_RePO_Normal_Test.ipynb`, `Flow_based_RePO_Adversarial.ipynb` e `Packet_based_RePO_Adversarial.ipynb`.

Os dados extraídos precisam existir em `data/`. Se necessário, prepare-os conforme as instruções do README; a extração não é executada automaticamente por este roteiro.